# PSP Jovian SVD Pipeline — End-to-End Demo

Demonstrates the full pipeline described in Wille et al. (in review),
stage by stage, calling into the `src/` package rather than redefining
functions inline. Requires the processed data products described in
`data/README.md` to be present (or `PSP_DATA_DIR` set).

This notebook is meant to be run top-to-bottom (Restart & Run All) as a
reproducibility check.

In [ ]:
import sys
sys.path.insert(0, "..")  # repo root, so `src` is importable

from src.preprocessing.load_data import load_jupiter_data, sort_frequency_channels
from src.svd_extraction.svd_core import run_svd_extraction, compare_svd_snr, analyze_spectrogram_snr
from src.svd_extraction.validation import compare_autocorrelations, plot_autocorr_power_spectrum_figure
from src.eigenfaces.eigenfaces_core import (
    compute_single_eigenface, project_single_eigenface,
    plot_eigenface_histogram, plot_eigenface_timeline,
)
from src.occurrence.occurrence_plots import plot_jovian_occurrence_matrix
from src.stokes_comparison.stokes_v import (
    filter_by_time_range, extract_stokes_v_columns, resample_stokes_to_phase_grid,
    compare_svd_to_stokes_v, full_baseline_comparison,
)
from src.stokes_comparison.pearson_plot import plot_stokes_v_pearson
from src.plotting.spectrogram_plots import plot_file_index, plot_spectral_smoothness
from src.config import RAW_MONTHLY_PATTERN

import pandas as pd

## 1. Load data (Section 2)

In [ ]:
datasets = load_jupiter_data()  # phase_frame="lambda_iii" by default
datasets = sort_frequency_channels(datasets)

## 2. SVD extraction on a single event (Section 3.2, Figure 3/8-11 style panels)

In [ ]:
FILE_INDEX = 245  # representative quiet-HOM event; swap for any valid index

svd_signal, residuals = run_svd_extraction(
    datasets=datasets,
    file_index=FILE_INDEX,
    channel_range=(30, 120),
    num_modes_to_check=15,
    entropy_limit=3.0,  # matches the paper's S_lim
    v_limit=2.0,
    mode="jupiter",
    emission_title="Quiet HOM Type",
)

## 3. SNR comparison (Section 4.1)

In [ ]:
raw_snr, svd_snr = compare_svd_snr(
    datasets=datasets,
    file_index=FILE_INDEX,
    mode="jupiter",
    noise_percentile=10,
    entropy_limit=3.0,
    vmin=0, vmax=25,
)

## 4. Autocorrelation / power-spectrum validation (Section 4.1, Figure 5)

In [ ]:
sub_freqs = datasets["freqs"].iloc[30:121]
sub_phase = datasets["phase"]

ac_svd, ac_res, ps_svd, ps_res, svd_lag1, res_lag1 = compare_autocorrelations(
    svd_signal, residuals, freqs=sub_freqs, phase=sub_phase,
)

In [ ]:
plot_autocorr_power_spectrum_figure(ac_svd, ac_res, ps_svd, ps_res, save_path="figure5.png")

## 5. Eigenfaces detection metric (Section 3.3 / 4.4, Figure 6)\n\nRequires a manually-curated `all_data_jupiter` subset with a `type` column (1=Quiet HOM, 2=Noisy HOM, 3=Vertex-Late) — see `src/eigenfaces/annotator.py`.

In [ ]:
eigenface = compute_single_eigenface(datasets, target_type=2)

proj_jupiter = project_single_eigenface(datasets["all_data_jupiter"], eigenface)
proj_total = project_single_eigenface(datasets["all_data_total"], eigenface)

plot_eigenface_timeline(proj_total, proj_jupiter)
plot_eigenface_histogram(proj_total, proj_jupiter)

## 6. Occurrence probability distributions (Section 4.3, Figure 4)

Loads both phase-folded archives (Phi_Io and lambda_III are two separate archives, not two columns of one -- see `src/preprocessing/load_data.py`'s `phase_frame` parameter). The rescaling constants in `src/occurrence/occurrence_plots.py` still need author confirmation (see its module docstring) before treating this as reproducing Figure 4 exactly. Requires the four annotation JSON files described in `data/README.md`.

In [ ]:
datasets_io_phase = load_jupiter_data(phase_frame="phi_io")
datasets_io_phase = sort_frequency_channels(datasets_io_phase)

datasets_longitude = datasets  # already loaded above with phase_frame="lambda_iii"

occ_all_neg, occ_jup_neg, occ_all_pos, occ_jup_pos = plot_jovian_occurrence_matrix(
    datasets_io_phase=datasets_io_phase,
    datasets_longitude=datasets_longitude,
    annotation_dir="../data",
    title="Jovian Radio Emission Occurrence Probability",
)

## 7. Stokes V cross-validation (Section 4.2)

Requires a monthly combined archive file (see `src/preprocessing/build_monthly_archive.py` and `data/README.md`). This example uses one representative window -- see the module docstring in `src/stokes_comparison/stokes_v.py` regarding scope.

In [ ]:
stokes = pd.read_hdf(RAW_MONTHLY_PATTERN.format(year=2020, month=6))

stokes_window = filter_by_time_range(stokes, "2020/06/15 07:30:19", "2020/06/15 17:25:26")
stokes_filtered = resample_stokes_to_phase_grid(extract_stokes_v_columns(stokes_window))

# Run the same SVD extraction on the raw Stokes V window for the
# apples-to-apples comparison (Section 4.2 control experiment):
stk_svd, stk_res = run_svd_extraction(
    standalone_matrix=stokes_filtered[30:121, :],
    standalone_freqs=datasets["freqs"].iloc[30:121],
    standalone_phase=datasets["phase"],
    entropy_limit=3.0,
    emission_title="Stokes V",
    plot=False,
)

r, p = compare_svd_to_stokes_v(svd_signal, stk_svd)
plot_stokes_v_pearson(svd_signal, stk_svd)